# 3. Training of the three candidate models (GPU)

Given D1 and D2, each model reconstructs A:

| name | model | task |
|---|---|---|
| `aligntag` | alignment-guided tagger (BiLSTM over the columns of a symmetric alignment of D1 and D2) | per column: A keeps the symbol of D1, keeps the symbol of D2, or has no copy there |
| `bilstm` | BiLSTM encoder, LSTM decoder with attention | [D1 ; SEP ; D2] → A, token by token |
| `transformer` | encoder-decoder Transformer | [D1 ; SEP ; D2] → A, token by token |

Variants are numbered within each family in order of first appearance, and every token carries the nucleotides of its variant, so that a model can see which variants are one substitution apart. Training families are augmented by exchanging D1 and D2 and, for the sequence-to-sequence models, by reversing all three sequences.

Model selection uses the validation set only: the mean relative error ε(A, Â) = e(A, Â) / n(A).

**Parallel use.** Set `MODELS` to one name per Colab session to train the three models at the same time (three GPU sessions), or leave the list to train them one after another. Training is checkpointed after every epoch and resumes if the session is interrupted. `STAGE = "pilot"` trains on 2,000 families for a few epochs for a first comparison; `STAGE = "full"` uses the whole training set.

Set the runtime to GPU (Runtime → Change runtime type) before running.

In [ ]:
#@title Setup: fetch the code and choose where results are kept
REPO_URL = "https://github.com/AtheerMatroud/TandemRepeats"  #@param {type:"string"}
SAVE_TO_DRIVE = True  #@param {type:"boolean"}

import os, sys, subprocess, zipfile, io
if not os.path.exists("/content/TandemRepeats/AI/tr_ai"):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "/content/TandemRepeats"])
if not os.path.exists("/content/TandemRepeats/AI/tr_ai"):
    from google.colab import files
    print("The AI folder was not found in the repository. Upload AI.zip instead.")
    up = files.upload(); name = next(iter(up))
    os.makedirs("/content/TandemRepeats", exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(up[name])) as z: z.extractall("/content/TandemRepeats")
AI = "/content/TandemRepeats/AI"
sys.path.insert(0, AI)
os.environ["PYTHONPATH"] = AI + ":" + os.environ.get("PYTHONPATH", "")
if SAVE_TO_DRIVE:
    from google.colab import drive; drive.mount("/content/drive")
    ROOT = "/content/drive/MyDrive/tandem_repeats/ai_experiment"
else:
    ROOT = "/content/ai_experiment"
os.makedirs(ROOT, exist_ok=True)
import json, glob, time, numpy as np, pandas as pd
print("code:", AI, "| results:", ROOT, "| CPU cores:", os.cpu_count())


In [ ]:
#@title Parameters
MODELS = ["aligntag", "bilstm", "transformer"]  # one name per session to train in parallel
STAGE = "full"  #@param ["pilot", "full"]
import torch
from tr_ai import simdata
from tr_ai.train import train
from tr_ai.models import DEFAULTS
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, torch.cuda.get_device_name(0) if device.type == "cuda" else "(no GPU: training will be very slow)")
fams = {f["id"]: f for f in simdata.load(f"{ROOT}/families/shard_*.jsonl.gz")}
splits = json.load(open(f"{ROOT}/splits.json"))
train_fams = [fams[i] for i in splits["train"]]; val_fams = [fams[i] for i in splits["val"]]
OVERRIDES = {}
if STAGE == "pilot":
    train_fams = train_fams[:2000]
    OVERRIDES = {m: {"epochs": 8} for m in MODELS}
print(len(train_fams), "training and", len(val_fams), "validation families")


In [ ]:
#@title Train (resumable)
for name in MODELS:
    out = f"{ROOT}/models/{STAGE}/{name}"
    if os.path.exists(f"{out}/done.json"):
        print(name, "already trained:", json.load(open(f"{out}/done.json"))["best"]); continue
    best = train(name, train_fams, val_fams, out, device, overrides=OVERRIDES.get(name), workers=os.cpu_count())
    print(name, "best on validation:", best)


In [ ]:
#@title Comparison of the candidates on the validation set
import matplotlib.pyplot as plt
rows = []
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
for p in sorted(glob.glob(f"{ROOT}/models/{STAGE}/*/log.csv")):
    name = p.split("/")[-2]; log = pd.read_csv(p)
    ax[0].plot(log.epoch, log.val_loss, label=name)
    v = log.dropna(subset=["val_eps"]); ax[1].plot(v.epoch, 100 * v.val_eps, marker="o", label=name)
    d = f"{ROOT}/models/{STAGE}/{name}/done.json"
    if os.path.exists(d):
        b = json.load(open(d)); rows.append({"model": name, "parameters (M)": round(b["parameters"] / 1e6, 1), "best epoch": b["best"]["epoch"],
                                              "val families": b["best"]["n_val"], "exact (%)": round(100 * b["best"]["exact"], 1),
                                              "mean e": round(b["best"]["edit"], 1), "mean eps (%)": round(100 * b["best"]["eps"], 2),
                                              "mean n(Ahat)-n(A)": round(b["best"]["excess"], 1),
                                              "minutes": round(log.seconds.sum() / 60, 1)})
ax[0].set_title("validation loss"); ax[1].set_title("validation mean relative error (%)"); ax[1].set_yscale("log")
for a in ax: a.set_xlabel("epoch"); a.grid(alpha=.3); a.legend()
plt.tight_layout(); plt.savefig(f"{ROOT}/models/{STAGE}_curves.png", dpi=150); plt.show()
if rows:
    cmp = pd.DataFrame(rows).sort_values("mean eps (%)"); display(cmp); cmp.to_csv(f"{ROOT}/models/{STAGE}_comparison.csv", index=False)
